<a href="https://colab.research.google.com/github/felimaker/IBM-Data-Science-Capstone/blob/main/6_folium_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6: Analítica Visual Interactiva - Mapa con Folium
**Proyecto:** Predicción de aterrizaje de la primera etapa del Falcon 9

**Objetivo:** Construir un mapa interactivo con `Folium` que muestre:
1. Marcadores de todos los sitios de lanzamiento de SpaceX.
2. Registros (marcadores en clúster) de cada lanzamiento, coloreados por resultado
   (éxito/fallo).
3. Análisis de proximidad: distancia de cada sitio a la costa, autopistas más
   cercanas y ciudades cercanas.

Repositorio de GitHub: **https://github.com/felimaker/IBM-Data-Science-Capstone**


In [2]:
!pip install folium -q

import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

# Cargamos el archivo directamente desde la URL de respaldo del curso
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(url)
df.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## 1. Marcadores de los sitios de lanzamiento
Cada sitio se ubica una sola vez con su latitud/longitud promedio.


In [3]:
launch_sites_df = df.groupby('LaunchSite', as_index=False).agg(
    {'Latitude': 'first', 'Longitude': 'first'})

nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row['Latitude'], row['Longitude']]
    circle = folium.Circle(
        coordinate, radius=1000, color='#d35400', fill=True
    ).add_child(folium.Popup(row['LaunchSite']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20), icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{row["LaunchSite"]}</b></div>'
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map


## 2. Registros de lanzamientos con marker cluster (coloreados por resultado)
Verde = aterrizaje exitoso (`Class` = 1). Rojo = aterrizaje fallido (`Class` = 0).


In [4]:
df['marker_color'] = df['Class'].apply(lambda x: 'green' if x == 1 else 'red')

marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for _, row in df.iterrows():
    marker = folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=f"{row['LaunchSite']} — Class: {row['Class']}"
    )
    marker_cluster.add_child(marker)

site_map


## 3. Análisis de proximidad
Calculamos la distancia (Haversine) desde un sitio de lanzamiento a puntos de
interés cercanos (costa, autopista, vía férrea, ciudad), y dibujamos líneas
`PolyLine` que representan esa distancia sobre el mapa.


In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Ejemplo: distancia del sitio CCAFS SLC-40 a la costa más cercana
launch_site_lat, launch_site_lon = 28.563197, -80.576820  # CCAFS SLC-40
coastline_lat, coastline_lon = 28.56367, -80.57163        # punto de costa cercano

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distancia a la costa: {distance_coastline:.2f} km")


Distancia a la costa: 0.51 km


In [6]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#252526;"><b>{distance_coastline:.2f} KM</b></div>'
    )
)
site_map.add_child(distance_marker)

coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

site_map


In [8]:
site_map.save('launch_sites_map.html')
print("Mapa guardado como launch_sites_map.html")


Mapa guardado como launch_sites_map.html


## Resumen del mapa interactivo
- **Marcadores de sitios**: los 4 sitios de lanzamiento de SpaceX (CCAFS SLC-40,
  CCAFS LC-40, KSC LC-39A, VAFB SLC-4E) ubicados con círculos y etiquetas.
- **Registros de lanzamientos**: cada lanzamiento individual agrupado en un
  `MarkerCluster`, coloreado en verde (éxito) o rojo (fallo), permitiendo ver
  visualmente qué sitios concentran más éxitos.
- **Análisis de proximidad**: se calculó la distancia Haversine desde un sitio de
  lanzamiento hasta la costa más cercana, visualizada con una línea (`PolyLine`) y
  una etiqueta de distancia en kilómetros. El mismo patrón se puede repetir para
  medir distancia a autopistas, vías férreas y ciudades.
- Archivo exportado: `launch_sites_map.html` (mapa interactivo independiente).

Repositorio de GitHub: **https://github.com/felimaker/IBM-Data-Science-Capstone**
